# Stage 5 Hybrid Variance 5/15 正式训练入口

本 Notebook 只负责薄编排：环境与数据预检、调用已冻结的生产 runner、单 cycle 诊断、artifact 检查以及 checkpoint/resume 验证。本流程不包含 B2/B3/B4、Validation 或 Stage 6。

## Cell 1 — 环境与设备

本单元格仅报告 Python、PyTorch 和 CUDA 就绪状态，不会改变已冻结的训练合同。

In [ ]:
import json
import platform
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

import torch

PROJECT_ROOT = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / 'factor_gfn').is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DEVICE = 'cuda:0'
SEED = 42
CUDA_READY = torch.cuda.is_available()
ENVIRONMENT = {
    'project_root': str(PROJECT_ROOT),
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': CUDA_READY,
    'gpu_name': torch.cuda.get_device_name(0) if CUDA_READY else None,
    'device': DEVICE,
    'seed': SEED,
}
print(json.dumps(ENVIRONMENT, ensure_ascii=False, indent=2), flush=True)
if not CUDA_READY:
    print('CUDA is unavailable: preflight may inspect CPU-safe contracts, but formal training remains blocked.', flush=True)

## Cell 2 — 冻结的 Hybrid 配置

正式新 run 从创建时起就固定为 5/15、K=16、共享单个条件 policy，且最大总预算为 100 cycles。该预算只是同一 run 允许到达的上限，不会触发自动训练；实际执行仍由 Cell 5 和 Cell 9 的手动门禁分段控制。

In [ ]:
from factor_gfn.gfn import (
    ExhaustiveRegistry,
    HybridVarianceTrainer,
    RealRewardDataConfig,
    RealRewardDataPaths,
    RealRewardProvider,
    TRAIN_CANDIDATE_ARTIFACT_FILENAME,
    build_real_reward_data_context,
    build_stage5_hybrid_variance_5_15_config,
    create_hybrid_variance_runner,
    resume_hybrid_variance_runner,
)

FORMAL_MAX_CYCLES = 100
K = 16
config = build_stage5_hybrid_variance_5_15_config(
    max_cycles=FORMAL_MAX_CYCLES,
    trajectories_per_batch=K,
    seed=SEED,
)
CONTRACT = {
    'max_cycles': config.training.max_cycles,
    'max_depth': config.search_space.max_depth,
    'max_nodes': config.search_space.max_nodes,
    'K': config.training.trajectories_per_batch,
    'conditions': config.resolved_condition_node_counts,
    'exact_conditions': config.objective.exact_tb_node_counts,
    'lpv_conditions': config.objective.lpv_node_counts,
    'policy_optimizer': config.training.optimizer,
    'policy_lr': config.training.learning_rate,
    'gradient_clip': config.training.model_gradient_clip_norm,
    'trajectories_per_cycle': config.training.trajectories_per_cycle,
    'optimizer_steps_per_cycle': config.training.optimizer_steps_per_cycle,
    'planned_total_optimizer_steps': config.training.total_optimizer_steps,
    'planned_total_trajectories': config.training.total_training_trajectories,
    'config_fingerprint': config.fingerprint(),
}
assert CONTRACT['max_cycles'] == 100
assert CONTRACT['max_depth'] == 5 and CONTRACT['max_nodes'] == 15
assert CONTRACT['K'] == 16
assert CONTRACT['conditions'] == tuple(range(1, 16))
assert CONTRACT['exact_conditions'] == (1, 2)
assert CONTRACT['lpv_conditions'] == tuple(range(3, 16))
assert CONTRACT['trajectories_per_cycle'] == 240
assert CONTRACT['optimizer_steps_per_cycle'] == 15
assert CONTRACT['planned_total_optimizer_steps'] == 1500
assert CONTRACT['planned_total_trajectories'] == 24000
print(json.dumps(CONTRACT, ensure_ascii=False, indent=2), flush=True)

## Cell 3 — 不启动训练的就绪预检

本单元格加载仅 Train 数据，初始化真实 Reward provider，试算一个非 exact 表达式，验证 N=1/2 registry 的只读复用，并在 optimizer step 0 初始化临时 runner/artifact。它绝不调用 `train_step` 或 `run_attempts`。

In [ ]:
from factor_gfn.grammar import Expression, get_action_id

DATA_CONFIG = RealRewardDataConfig()
DATA_PATHS = RealRewardDataPaths()
SOURCE_REGISTRY = (
    PROJECT_ROOT
    / 'runs'
    / 'complexity_diagnostic_6_20'
    / 'manual_diagnostic_6_20_seed42'
    / 'exhaustive_registry.sqlite3'
)
RUN_ROOT = PROJECT_ROOT / 'runs' / 'stage5_hybrid_variance_real_5_15'

required_paths = [
    DATA_PATHS.tensor_path,
    DATA_PATHS.universe_mask_path,
    DATA_PATHS.date_list_path,
    DATA_PATHS.stock_list_path,
    DATA_PATHS.processed_metadata_path,
    DATA_PATHS.industry_path,
    DATA_PATHS.industry_metadata_path,
    DATA_PATHS.barra_paths.metadata_path,
    DATA_PATHS.barra_paths.market_return_path,
    *[DATA_PATHS.barra_paths.exposure_path(name) for name in ('market_beta', 'size', 'momentum', 'volatility', 'liquidity')],
    SOURCE_REGISTRY,
]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError({'missing_preflight_paths': missing_paths})

preflight_started = perf_counter()
context = build_real_reward_data_context(DATA_CONFIG, DATA_PATHS)
provider = RealRewardProvider(context, config.reward)
probe = Expression.from_prefix((get_action_id('add'), get_action_id('close'), get_action_id('open')))
probe_assignment = provider.evaluate(probe)
if provider.interpreter_evaluation_count != 1:
    raise RuntimeError('FactorInterpreter preflight did not perform exactly one evaluation')

preflight_device = DEVICE if CUDA_READY else 'cpu'
preflight_trainer = HybridVarianceTrainer(config, provider, device=preflight_device)
preflight_registry = ExhaustiveRegistry(SOURCE_REGISTRY, read_only=True)
try:
    exact_semantics = preflight_trainer.target_exhaustive_reuse_semantics()
    exact_proofs = preflight_trainer.configure_hybrid_exhaustive_registry(
        preflight_registry,
        source_semantics_by_N={1: exact_semantics, 2: exact_semantics},
    )
    with TemporaryDirectory(prefix='factor_gfn_hybrid_preflight_') as temporary:
        temporary_runner = create_hybrid_variance_runner(
            preflight_trainer,
            Path(temporary) / 'runner',
        )
        temporary_artifact = json.loads(
            temporary_runner.train_candidate_artifact_path.read_text(encoding='utf-8')
        )
        artifact_ready = (
            temporary_artifact['committed_optimizer_step'] == 0
            and temporary_artifact['candidate_count'] == 0
            and temporary_runner.latest_checkpoint_path.is_file()
        )
finally:
    preflight_registry.close()

PREFLIGHT = {
    'data_ready': True,
    'requested_train_range': [DATA_CONFIG.train_start, DATA_CONFIG.train_end],
    'actual_train_range': [context.manifest['actual_train_start'], context.manifest['actual_train_end']],
    'evaluation_shape': context.manifest['shape']['evaluation'],
    'rebalance_periods': context.manifest['calendar']['rebalance_periods'],
    'provider_ready': True,
    'provider_fingerprint': provider.fingerprint(),
    'validation_oos_loaded': provider.manifest()['validation_oos_loaded'],
    'factor_interpreter_probe_valid': probe_assignment.valid,
    'factor_interpreter_evaluations': provider.interpreter_evaluation_count,
    'exact_registry_read_only': preflight_registry.read_only,
    'exact_conditions_ready': sorted(exact_proofs),
    'exact_proof_fingerprints': {n: proof.proof_fingerprint for n, proof in exact_proofs.items()},
    'cuda_ready': CUDA_READY,
    'artifact_writer_ready': artifact_ready,
    'checkpoint_output_root': str(RUN_ROOT),
    'artifact_filename': TRAIN_CANDIDATE_ARTIFACT_FILENAME,
    'optimizer_step_after_preflight': preflight_trainer.optimizer_step,
    'preflight_seconds': perf_counter() - preflight_started,
}
assert PREFLIGHT['validation_oos_loaded'] is False
assert PREFLIGHT['exact_conditions_ready'] == [1, 2]
assert PREFLIGHT['artifact_writer_ready']
assert PREFLIGHT['optimizer_step_after_preflight'] == 0
PREFLIGHT_READY = all((
    PREFLIGHT['data_ready'],
    PREFLIGHT['provider_ready'],
    PREFLIGHT['cuda_ready'],
    PREFLIGHT['artifact_writer_ready'],
    PREFLIGHT['exact_conditions_ready'] == [1, 2],
))
print(json.dumps(PREFLIGHT, ensure_ascii=False, indent=2), flush=True)
print({'PREFLIGHT_READY': PREFLIGHT_READY, 'REAL_TRAINING_EXECUTED': False}, flush=True)

## Cell 4 — 受门禁保护的 new/resume runner 构建

在人工审阅预检报告前，保持 `RUN_REAL_ONE_CYCLE=False`。构建新 runner 只会写入初始 step-0 checkpoint/state/artifact，不会执行训练。任何会改变配置 fingerprint 的调参都必须使用 `MODE='new'` 创建全新 run，不得强制 resume 旧 checkpoint。

In [ ]:
RUN_REAL_ONE_CYCLE = False
MODE = 'resume'
RESUME_RUN_DIR = r'D:\实习\Gflownet因子挖掘\runs\stage5_hybrid_variance_real_5_15\hybrid_5_15_k16_seed42_20260816T025559Z'

if not RUN_REAL_ONE_CYCLE:
    raise RuntimeError('Safety stop: review preflight, then explicitly set RUN_REAL_ONE_CYCLE=True')
if not PREFLIGHT_READY:
    raise RuntimeError('Formal one-cycle run is blocked because preflight is not ready')
if MODE not in {'new', 'resume'}:
    raise ValueError("MODE must be 'new' or 'resume'")
if MODE == 'new' and RESUME_RUN_DIR is not None:
    raise ValueError('new mode must not set RESUME_RUN_DIR')
if MODE == 'resume' and RESUME_RUN_DIR is None:
    raise ValueError('resume mode requires an explicit RESUME_RUN_DIR')

formal_context = build_real_reward_data_context(DATA_CONFIG, DATA_PATHS)
formal_provider = RealRewardProvider(formal_context, config.reward)
formal_trainer = HybridVarianceTrainer(config, formal_provider, device=DEVICE)
exact_registry = ExhaustiveRegistry(SOURCE_REGISTRY, read_only=True)
formal_semantics = formal_trainer.target_exhaustive_reuse_semantics()
formal_trainer.configure_hybrid_exhaustive_registry(
    exact_registry,
    source_semantics_by_N={1: formal_semantics, 2: formal_semantics},
)
if MODE == 'new':
    run_id = 'hybrid_5_15_k16_seed42_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    runner = create_hybrid_variance_runner(formal_trainer, RUN_ROOT / run_id)
else:
    runner = resume_hybrid_variance_runner(Path(RESUME_RUN_DIR), formal_trainer)
print({
    'mode': MODE,
    'run_dir': str(runner.run_dir),
    'optimizer_step': runner.trainer.optimizer_step,
    'cycle_assignment': asdict(runner.trainer.complexity_scheduler.peek()),
    'checkpoint': str(runner.latest_checkpoint_path),
    'artifact': str(runner.train_candidate_artifact_path),
}, flush=True)

## Cell 5 — 严格只执行首个 cycle

本单元格只允许从全新 step-0 run 开始，并在 15 次成功 optimizer 更新、240 条已接受 trajectory 后停止。它会记录每个条件的耗时和可用的 CUDA 内存信息。

In [ ]:
if not RUN_REAL_ONE_CYCLE:
    raise RuntimeError('Safety stop: one-cycle execution is disabled')
if runner.trainer.optimizer_step != 0:
    raise RuntimeError('Initial one-cycle cell requires a fresh step-0 runner and must not be rerun')
cycle_started = perf_counter()
start_assignment = runner.trainer.complexity_scheduler.peek()
start_cycle = start_assignment.cycle_index
start_optimizer_step = runner.trainer.optimizer_step
batch_runtime = []
if CUDA_READY:
    torch.cuda.reset_peak_memory_stats()
while not runner.complete and runner.trainer.complexity_scheduler.peek().cycle_index == start_cycle:
    batch_started = perf_counter()
    output = runner.run_attempts(1)[0]
    elapsed = perf_counter() - batch_started
    if output.updated:
        batch_runtime.append({
            'condition_N': output.diagnostics.condition_N,
            'optimizer_step': output.global_optimizer_step,
            'elapsed_seconds': elapsed,
            'cuda_allocated_bytes': torch.cuda.memory_allocated() if CUDA_READY else None,
            'cuda_peak_bytes': torch.cuda.max_memory_allocated() if CUDA_READY else None,
        })
        print(batch_runtime[-1], flush=True)
cycle_runtime_seconds = perf_counter() - cycle_started
successful_updates = runner.trainer.optimizer_step - start_optimizer_step
assert successful_updates == config.training.optimizer_steps_per_cycle
assert runner.trainer.total_trajectories_seen == config.training.trajectories_per_cycle
print({'successful_updates': successful_updates, 'cycle_runtime_seconds': cycle_runtime_seconds, 'run_dir': str(runner.run_dir)}, flush=True)

## Cell 6 — 分目标诊断与运行时间

本单元格只读展示按条件 N 分组的 exact-TB/LPV、Reward、梯度、重试与多样性诊断，不会更新模型。

In [ ]:
diagnostics_by_N = {item.condition_N: item.to_dict() for item in runner.trainer.diagnostic_history[-15:]}
exact_fields = ('exact_log_z', 'tb_loss', 'tb_delta_mean', 'tb_delta_std', 'tb_delta_rms')
lpv_fields = ('zeta_mean', 'zeta_std', 'zeta_variance', 'variance_loss', 'centered_zeta_rms', 'unique_terminal_count', 'unique_terminal_fraction')
common_fields = ('reward_mean', 'sum_log_pf_mean', 'sum_log_pb_mean', 'policy_grad_norm', 'requested_count', 'accepted_count', 'invalid_count', 'retry_count', 'retry_exhausted_count', 'trajectories_in_batch', 'global_optimizer_step', 'condition_position_in_cycle')
for condition_N in sorted(diagnostics_by_N):
    row = diagnostics_by_N[condition_N]
    objective_fields = exact_fields if condition_N in (1, 2) else lpv_fields
    print({'condition_N': condition_N, **{name: row[name] for name in (*objective_fields, *common_fields)}}, flush=True)
print({'per_N_runtime': batch_runtime, 'full_cycle_runtime_seconds': cycle_runtime_seconds}, flush=True)

## Cell 7 — Train artifact 检查

本单元格只验证 B1 输出及 artifact committed step，不会运行 Stage 6。

In [ ]:
artifact = json.loads(runner.train_candidate_artifact_path.read_text(encoding='utf-8'))
assert artifact['committed_optimizer_step'] == runner.trainer.optimizer_step
assert artifact['candidate_count'] == len(artifact['records'])
sample_fields = ('structural_hash', 'train_ic', 'train_direction', 'train_long_ir', 'train_long_excess_dates', 'train_long_excess_values', 'train_barra_ts_corr', 'train_barra_correlations', 'train_barra_valid_periods_by_style', 'first_seen', 'last_seen', 'visit_count')
sample = [{name: record[name] for name in sample_fields} for record in artifact['records'][:3]]
print({'artifact_path': str(runner.train_candidate_artifact_path), 'committed_optimizer_step': artifact['committed_optimizer_step'], 'candidate_count': artifact['candidate_count'], 'sample': sample}, flush=True)

## Cell 8 — 仅验证 checkpoint/resume

本单元格重建与当前 fingerprint 匹配的 provider/trainer，加载已保存 run，并核对 policy、optimizer step、scheduler、trajectory 计数和 artifact step。它本身不会继续训练。

In [ ]:
saved = {
    'model': {name: value.detach().cpu().clone() for name, value in runner.trainer.model.state_dict().items()},
    'optimizer_step': runner.trainer.optimizer_step,
    'total_trajectories_seen': runner.trainer.total_trajectories_seen,
    'scheduler': runner.trainer.complexity_scheduler.state_dict(),
    'artifact_step': artifact['committed_optimizer_step'],
}
resume_context = build_real_reward_data_context(DATA_CONFIG, DATA_PATHS)
resume_provider = RealRewardProvider(resume_context, config.reward)
resume_trainer = HybridVarianceTrainer(config, resume_provider, device=DEVICE)
resume_registry = ExhaustiveRegistry(SOURCE_REGISTRY, read_only=True)
resume_semantics = resume_trainer.target_exhaustive_reuse_semantics()
resume_trainer.configure_hybrid_exhaustive_registry(resume_registry, source_semantics_by_N={1: resume_semantics, 2: resume_semantics})
resumed_runner = resume_hybrid_variance_runner(runner.run_dir, resume_trainer)
assert resumed_runner.trainer.optimizer_step == saved['optimizer_step']
assert resumed_runner.trainer.total_trajectories_seen == saved['total_trajectories_seen']
assert resumed_runner.trainer.complexity_scheduler.state_dict() == saved['scheduler']
assert all(torch.equal(value.detach().cpu(), saved['model'][name]) for name, value in resumed_runner.trainer.model.state_dict().items())
resumed_artifact = json.loads(resumed_runner.train_candidate_artifact_path.read_text(encoding='utf-8'))
assert resumed_artifact['committed_optimizer_step'] == saved['artifact_step']
print({'resume_verified': True, 'optimizer_step': resumed_runner.trainer.optimizer_step, 'trajectories': resumed_runner.trainer.total_trajectories_seen, 'artifact_step': resumed_artifact['committed_optimizer_step']}, flush=True)

## Cell 9 — 训练到指定累计 cycle 的通用续训入口（手动门禁）

本单元格按“累计训练到目标 cycle”控制同一 run，而不是追加一个需手算的 cycle 数。首轮验证后将 `TARGET_CYCLE=6`，会从累计 1 cycle 继续到 6；若人工检查后不调参，将其改为 `TARGET_CYCLE=100` 即会从当前累计进度继续到 100。续训完成后可重跑 Cell 6–8 检查最新诊断、artifact 和 resume。保存的 Notebook 中门禁始终默认关闭。

In [ ]:
TARGET_CYCLE = 100
RUN_TO_TARGET_CYCLE = False

def run_until_cycle(active_runner, target_cycle):
    if isinstance(target_cycle, bool) or not isinstance(target_cycle, int) or target_cycle < 1:
        raise ValueError('target_cycle must be a positive integer')
    training = active_runner.trainer.config.training
    if target_cycle > training.max_cycles:
        raise ValueError(f'target_cycle={target_cycle} exceeds the frozen run budget {training.max_cycles}')
    steps_per_cycle = training.optimizer_steps_per_cycle
    start_optimizer_step = active_runner.trainer.optimizer_step
    start_cycle = start_optimizer_step // steps_per_cycle
    start_position_in_cycle = start_optimizer_step % steps_per_cycle
    target_optimizer_step = target_cycle * steps_per_cycle
    if target_optimizer_step < start_optimizer_step:
        raise ValueError(f'target_cycle={target_cycle} is behind current optimizer step {start_optimizer_step}')
    attempted_updates = 0
    successful_updates = 0
    continuation_started = perf_counter()
    while active_runner.trainer.optimizer_step < target_optimizer_step:
        if active_runner.complete:
            raise RuntimeError('runner became complete before reaching target_cycle')
        update_started = perf_counter()
        output = active_runner.run_attempts(1)[0]
        attempted_updates += 1
        if output.updated:
            successful_updates += 1
            print({
                'cycle_index': output.diagnostics.cycle_index,
                'condition_N': output.diagnostics.condition_N,
                'optimizer_step': output.global_optimizer_step,
                'elapsed_seconds': perf_counter() - update_started,
                'objective_kind': output.diagnostics.objective_kind,
                'policy_grad_norm_pre_clip': output.diagnostics.policy_grad_norm,
            }, flush=True)
    expected_updates = target_optimizer_step - start_optimizer_step
    if active_runner.trainer.optimizer_step - start_optimizer_step != expected_updates:
        raise RuntimeError('optimizer-step advance differs from the requested cumulative target cycle')
    if active_runner.trainer.total_trajectories_seen != target_cycle * training.trajectories_per_cycle:
        raise RuntimeError('trajectory count differs from the requested cumulative target cycle')
    final_artifact = json.loads(active_runner.train_candidate_artifact_path.read_text(encoding='utf-8'))
    if final_artifact['committed_optimizer_step'] != active_runner.trainer.optimizer_step:
        raise RuntimeError('artifact committed step differs from the runner optimizer step')
    summary = {
        'start_cycle': start_cycle,
        'start_position_in_cycle': start_position_in_cycle,
        'target_cycle': target_cycle,
        'equivalent_cycles_completed_this_call': expected_updates / steps_per_cycle,
        'attempts': attempted_updates,
        'successful_optimizer_updates': successful_updates,
        'final_optimizer_step': active_runner.trainer.optimizer_step,
        'final_trajectories': active_runner.trainer.total_trajectories_seen,
        'final_candidate_count': final_artifact['candidate_count'],
        'runner_complete': active_runner.complete,
        'continuation_seconds': perf_counter() - continuation_started,
        'run_dir': str(active_runner.run_dir),
    }
    print(summary, flush=True)
    return summary

if RUN_TO_TARGET_CYCLE:
    active_runner = resumed_runner if 'resumed_runner' in globals() else runner
    continuation_summary = run_until_cycle(active_runner, target_cycle=TARGET_CYCLE)
    runner = active_runner
else:
    print({'continuation_enabled': False, 'target_cycle': TARGET_CYCLE}, flush=True)